In [ ]:
from pathlib import Path

import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

OUT = Path("results/hypothesis")
OUT.mkdir(parents=True, exist_ok=True)
FIG = Path("figures")
FIG.mkdir(parents=True, exist_ok=True)
BLOCK = 10
N_BOOT = 2000
HOLM_P_FOUR_TESTS = 0.0068143100511073255
DOMAIN_COLOR = {
    "2.60.40.1730": "#0072B2",
    "3.30.2010.30": "#E69F00",
    "1.10.390.10": "#009E73",
    "1.25.40.320": "#CC79A7",
    "none": "#7F7F7F",
}
DOMAIN_LABEL = {
    "2.60.40.1730": "Ig-like",
    "3.30.2010.30": "zincin-like",
    "1.10.390.10": "protease domain 2",
    "1.25.40.320": "C-terminal M1",
    "none": "outside CATH",
}
sites = pd.read_csv("results/rates/sites_with_structure.csv").sort_values("uniprot_position")
y = np.log1p(sites.relative_rate.to_numpy(dtype=float))
distance = sites.min_heavy_atom_distance_to_functional_A.to_numpy(dtype=float)
rsa = sites.rsa.to_numpy(dtype=float)
contacts = sites.n_contacts.to_numpy(dtype=float)


def zscore(values):
    center = float(values.mean())
    scale = float(values.std(ddof=0))
    return (values - center) / scale, center, scale


z_rsa, _, _ = zscore(rsa)
z_contacts, _, _ = zscore(contacts)
z_distance, distance_center, distance_scale = zscore(distance)
full = np.column_stack([np.ones(len(y)), z_rsa, z_contacts, z_distance])
reduced = full[:, :3]
beta = np.linalg.lstsq(full, y, rcond=None)[0]
reduced_beta = np.linalg.lstsq(reduced, y, rcond=None)[0]
residual = y - reduced @ reduced_beta
observed = float(beta[3])
count = 1
for shift in range(1, len(y)):
    permuted = reduced @ reduced_beta + np.roll(residual, shift)
    value = float(np.linalg.lstsq(full, permuted, rcond=None)[0][3])
    count += int(abs(value) >= abs(observed) - 1e-8)
permutation_p = count / len(y)
rss = lambda response, design: float(np.sum((response - design @ np.linalg.lstsq(design, response, rcond=None)[0]) ** 2))
partial_r2 = (rss(y, reduced) - rss(y, full)) / rss(y, reduced)
rng = np.random.default_rng(12345)
samples = []
attempts = 0
while len(samples) < N_BOOT:
    attempts += 1
    if attempts > N_BOOT + 800:
        raise SystemExit("block bootstrap produced too many singular resamples")
    n_blocks = int(np.ceil(len(y) / BLOCK))
    starts = rng.integers(0, len(y), size=n_blocks)
    index = (starts[:, None] + np.arange(BLOCK)[None, :]) % len(y)
    index = index.ravel()[: len(y)]
    if any(column > 0 and float(np.std(full[index, column])) < 1e-10 for column in (1, 2, 3)):
        continue
    samples.append(float(np.linalg.lstsq(full[index], y[index], rcond=None)[0][3]))
low, high = np.quantile(np.asarray(samples[:N_BOOT]), [0.025, 0.975])
partial_residual = (y - full @ beta) + observed * z_distance
saved = sites[["uniprot_position", "cath_id", "rsa", "n_contacts"]].copy()
saved["distance_A"] = distance
saved["z_distance"] = z_distance
saved["partial_residual"] = partial_residual
saved.to_csv(OUT / "h2_partial_residuals.csv", index=False)
pd.DataFrame([{
    "n": len(y),
    "beta_distance": observed,
    "partial_r2": partial_r2,
    "permutation_p": permutation_p,
    "block_bootstrap_ci95_low": float(low),
    "block_bootstrap_ci95_high": float(high),
    "block_length": BLOCK,
    "bootstrap_replicates": N_BOOT,
    "bootstrap_seed": 12345,
    "distance_mean_A": distance_center,
    "distance_sd_A": distance_scale,
    "holm_p_four_tests": HOLM_P_FOUR_TESTS,
}]).to_csv(OUT / "h2_distance.csv", index=False)
plt.rcParams.update({"font.family": "DejaVu Sans"})
figure, axis = plt.subplots(figsize=(6.67, 7.5))
for ident, frame in saved.groupby("cath_id", sort=False):
    axis.scatter(frame.distance_A, frame.partial_residual, s=36, alpha=0.55, c=DOMAIN_COLOR[ident], label=DOMAIN_LABEL[ident])
grid = np.linspace(distance.min(), distance.max(), 200)
z_grid = (grid - distance_center) / distance_scale
axis.plot(grid, observed * z_grid, color="#222222", lw=2.6, label=f"Partial fit, beta = {observed:+.3f} per 1 SD ({distance_scale:.2f} A)")
axis.fill_between(grid, low * z_grid, high * z_grid, color="#222222", alpha=0.15, label="95% block-bootstrap interval of the slope")
axis.set_xlabel("Distance to functional center (A)")
axis.set_ylabel("Adjusted evolutionary rate (partial residual)")
axis.set_title("Evolutionary rates increase with distance\nfrom the functional center")
axis.legend(loc="upper center", bbox_to_anchor=(0.5, -0.12), ncol=2, frameon=False, fontsize=9)
figure.subplots_adjust(left=0.16, right=0.97, bottom=0.24, top=0.90)
figure.savefig(FIG / "h2_partial_effect.png", dpi=300)
plt.close()
